In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
from pathlib import Path

In [2]:
# ============================================================
# PATHS
# ============================================================
UCDB_GPKG = Path(
    r"D:\VSG\DIRTY_MODEL\February2026\GHSUCDB_Analysis\GHS_STAT_UCDB2015MT_GLOBE_R2019A\GHS_STAT_UCDB2015MT_GLOBE_R2019A_V1_2_with_GHSPOP2023.gpkg"
)

CITY_QC_CSV = Path(
    r"D:\VSG\DIRTY_MODEL\February2026\GHSUCDB_Analysis\city_deprivation_80pct_qc_with_ucdb_regions.csv"
)

OUT_DIR = Path(
    r"D:\VSG\DIRTY_MODEL\February2026\GHSUCDB_Analysis"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
# ============================================================
# COLUMN NAMES
# ============================================================
UCDB_ID_COL = "ID_HDC_G0"
UCDB_POP_COL = "GHSPOP2023"
UCDB_COUNTRY_COL = "CTR_MN_NM"

TARGET_REGIONS = ["Africa", "Asia", "Latin America and the Caribbean"]

In [4]:
ucdb_layer = gpd.list_layers(UCDB_GPKG).iloc[0]["name"]

ucdb = gpd.read_file(
    UCDB_GPKG,
    layer=ucdb_layer,
    columns=[UCDB_ID_COL, UCDB_COUNTRY_COL, UCDB_POP_COL, "GRGN_L1"]
)

ucdb[UCDB_ID_COL] = ucdb[UCDB_ID_COL].astype(str)

# Unique cities
ucdb_unique = (
    ucdb
    .dropna(subset=[UCDB_ID_COL])
    .drop_duplicates(subset=[UCDB_ID_COL])
    .copy()
)

# Filter regions
ucdb_unique = ucdb_unique[
    ucdb_unique["GRGN_L1"].isin(TARGET_REGIONS)
].copy()

print("UCDB shape:", ucdb_unique.shape)
ucdb_unique.head()

UCDB shape: (11618, 5)


,CTR_MN_NM,GHSPOP2023,GRGN_L1,ID_HDC_G0,geometry
18,Mexico,78430.922864,Latin America and the Caribbean,19.0,"MULTIPOLYGON (((-117.0657 32.42185, -117.05462..."
19,Mexico,332786.166521,Latin America and the Caribbean,20.0,"MULTIPOLYGON (((-116.60026 31.92206, -116.5892..."
22,Mexico,56171.304272,Latin America and the Caribbean,23.0,"MULTIPOLYGON (((-116.98401 32.43048, -116.9507..."
30,Mexico,69365.726105,Latin America and the Caribbean,31.0,"MULTIPOLYGON (((-116.67019 32.57717, -116.5925..."
42,Mexico,253371.783538,Latin America and the Caribbean,43.0,"MULTIPOLYGON (((-109.95075 22.94636, -109.9192..."


In [5]:
city_df = pd.read_csv(CITY_QC_CSV)

city_df[UCDB_ID_COL] = city_df[UCDB_ID_COL].astype(str)

print("City QC table:", city_df.shape)
city_df.head()

City QC table: (5204, 15)


,ID_HDC_G0,Region,Region_L2_pred,Country,City,TotalBlocks,ValidBlocks,PctValid,TotalPop,DeprivedPop,PctDeprived,Region_L1,Region_L2,UCDB_CityPop,CitySizeClass
0,5959.0,Asia,South-Central Asia,afghanistan,Herat,775,775,100.0,1.333792e+06,151072.083843,11.326507,Asia,South-Central Asia,1.333795e+06,Large
1,5964.0,Asia,South-Central Asia,afghanistan,Guzarah,71,71,100.0,1.647973e+05,0.000000,0.000000,Asia,South-Central Asia,1.647976e+05,Small
2,5973.0,Asia,South-Central Asia,afghanistan,Farah,124,124,100.0,1.353276e+05,0.000000,0.000000,Asia,South-Central Asia,1.353277e+05,Small
3,5981.0,Asia,South-Central Asia,afghanistan,Zaranj,44,44,100.0,3.893846e+04,0.000000,0.000000,Asia,South-Central Asia,3.893846e+04,Small
4,5992.0,Asia,South-Central Asia,afghanistan,Maymana,147,147,100.0,1.170994e+05,0.000000,0.000000,Asia,South-Central Asia,1.170994e+05,Small


In [6]:
city_df = city_df.merge(
    ucdb_unique[[UCDB_ID_COL, UCDB_COUNTRY_COL]],
    on=UCDB_ID_COL,
    how="left"
)

print("After merge:", city_df.shape)

# Check missing countries
missing = city_df[UCDB_COUNTRY_COL].isna().sum()
print("Missing country assignments:", missing)

After merge: (5204, 16)
Missing country assignments: 0


In [7]:
ucdb_country = (
    ucdb_unique
    .groupby(UCDB_COUNTRY_COL, as_index=False)
    .agg(UCDB_Pop=(UCDB_POP_COL, "sum"))
)

ucdb_country["UCDB_Pop_M"] = ucdb_country["UCDB_Pop"] / 1e6

ucdb_country.head()

,CTR_MN_NM,UCDB_Pop,UCDB_Pop_M
0,Afghanistan,1.439570e+07,14.395698
1,Algeria,1.953966e+07,19.539665
2,Angola,2.150803e+07,21.508028
3,Argentina,2.848656e+07,28.486560
4,Armenia,1.168163e+06,1.168163


In [8]:
cs_country = (
    city_df
    .groupby(UCDB_COUNTRY_COL, as_index=False)
    .agg(
        CS_Pop=("TotalPop", "sum"),
        CS_DeprivedPop=("DeprivedPop", "sum")
    )
)

cs_country["CS_Pop_M"] = cs_country["CS_Pop"] / 1e6

cs_country.head()

,CTR_MN_NM,CS_Pop,CS_DeprivedPop,CS_Pop_M
0,Afghanistan,1.151777e+07,8.037359e+05,11.517774
1,Algeria,1.932588e+07,2.148402e+06,19.325879
2,Angola,2.065083e+07,1.261948e+07,20.650830
3,Argentina,2.839333e+07,2.023375e+06,28.393329
4,Bangladesh,6.854272e+07,1.402888e+07,68.542720


In [9]:
country_table = pd.merge(
    ucdb_country[[UCDB_COUNTRY_COL, "UCDB_Pop_M"]],
    cs_country[[UCDB_COUNTRY_COL, "CS_Pop_M"]],
    on=UCDB_COUNTRY_COL,
    how="left"
)

# Fill missing (countries not covered by CS)
country_table["CS_Pop_M"] = country_table["CS_Pop_M"].fillna(0)

# Omitted
country_table["Omitted_Pop_M"] = (
    country_table["UCDB_Pop_M"] - country_table["CS_Pop_M"]
).clip(lower=0)

country_table["Omitted_%"] = np.where(
    country_table["UCDB_Pop_M"] > 0,
    (country_table["Omitted_Pop_M"] / country_table["UCDB_Pop_M"]) * 100,
    0
)

country_table["Omitted_%"] = country_table["Omitted_%"].round(1)

country_table.head()

,CTR_MN_NM,UCDB_Pop_M,CS_Pop_M,Omitted_Pop_M,Omitted_%
0,Afghanistan,14.395698,11.517774,2.877923,20.0
1,Algeria,19.539665,19.325879,0.213785,1.1
2,Angola,21.508028,20.650830,0.857198,4.0
3,Argentina,28.486560,28.393329,0.093231,0.3
4,Armenia,1.168163,0.000000,1.168163,100.0


In [10]:
country_table = country_table.rename(columns={
    UCDB_COUNTRY_COL: "Country"
})

# Sort by omitted population (most important)
country_table = country_table.sort_values(
    "Omitted_Pop_M", ascending=False
).reset_index(drop=True)

# Round population columns
for col in ["UCDB_Pop_M", "CS_Pop_M", "Omitted_Pop_M"]:
    country_table[col] = country_table[col].round(1)

print(country_table.head(20))

                             Country  UCDB_Pop_M  CS_Pop_M  Omitted_Pop_M  \
0                              China       636.4       0.0          636.4   
1                              Japan        84.5       0.0           84.5   
2                              India       559.8     479.7           80.0   
3                             Turkey        52.6       0.0           52.6   
4                        South Korea        38.6       0.0           38.6   
5                       Saudi Arabia        24.6       0.0           24.6   
6                         Bangladesh        90.1      68.5           21.6   
7                           Ethiopia        34.7      15.0           19.7   
8                             Taiwan        19.0       0.0           19.0   
9                              Chile        12.1       0.0           12.1   
10                           Nigeria       100.4      91.6            8.8   
11                         Singapore         7.6       0.0            7.6   

In [11]:
out_csv = OUT_DIR / "country_level_omission_table.csv"
country_table.to_csv(out_csv, index=False)

print("Saved to:", out_csv)

Saved to: D:\VSG\DIRTY_MODEL\February2026\GHSUCDB_Analysis\country_level_omission_table.csv


In [12]:
print("Total UCDB population:", country_table["UCDB_Pop_M"].sum())
print("Total CS population:", country_table["CS_Pop_M"].sum())
print("Total omitted population:", country_table["Omitted_Pop_M"].sum())

Total UCDB population: 3056.9
Total CS population: 1958.5
Total omitted population: 1098.5000000000002
